In [46]:
import json
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from typing import TypedDict,List
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel,Field
from typing import List , Dict , Any



In [47]:

class GraphState(BaseModel):
    user_query: str = Field(..., description="Input from the user")
    sql_query: str = Field(..., description="Response generated by the model")
    has_problem: bool = Field(..., description="True if there's a problem with the SQL query, False otherwise")
    problem_description: str = Field(..., description="Description of the problem if has_problem is True (empty string if no problem)")


class OutputFormat(BaseModel):
    has_problem: bool = Field(..., description="True if there's a problem with the SQL query, False otherwise")
    problem_description: str = Field(..., description="Description of the problem if has_problem is True (empty string if no problem)")


In [48]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="qwen3-vl:235b-cloud",
    temperature=0.5
)

In [49]:
structured_llm  = llm.with_structured_output(OutputFormat)


In [50]:
prompt_generator = ChatPromptTemplate.from_messages([
    ("system", """
You're a security-conscious SQL expert. Evaluate if the provided SQL query is dangerous (e.g., could delete/drop tables, expose sensitive data, or cause denial-of-service).

INSTRUCTIONS:
1. ALWAYS return valid JSON
2. has_problem MUST be a boolean (true/false)
3. problem_description MUST be an empty string "" when has_problem is false
4. NEVER use null/None for problem_description
5. If no problem exists: {{"has_problem": false, "problem_description": ""}}
6. If problem exists: {{"has_problem": true, "problem_description": "EXACT ISSUE HERE"}}

Check for:
- Destructive operations (DROP, DELETE without WHERE, TRUNCATE)
- Excessive data exposure (SELECT * on large tables)
- Privilege escalation attempts
- Time-consuming operations (no LIMIT on large queries)
"""),
    ("human", "User query: {user_query}\nSQL query: {sql_query}")
])


In [51]:
def evaluator(state: GraphState) -> dict:
    response: OutputFormat = structured_llm.invoke(
        prompt_generator.format_messages(
            user_query=state.user_query, 
            sql_query=state.sql_query
        )
    )
    # Return dictionary with correct keys matching GraphState fields
    return {
        "has_problem": response.has_problem,
        "problem_description": response.problem_description
    }

In [52]:
from langgraph.graph import StateGraph, END

graph = StateGraph(GraphState)

graph.add_node("evaluator", evaluator)

# Entry point must be ONE
graph.set_entry_point("evaluator")

# Generator → Evaluator
graph.add_edge("evaluator", END)

agent = graph.compile()

In [53]:
agent.invoke({
    "user_query": "give me email id of aarav sharma in users table",
    "sql_query": "SELECT email FROM users WHERE name = 'Aarav Sharma';",
    "has_problem": False,
    "problem_description": ""
})


{'user_query': 'give me email id of aarav sharma in users table',
 'sql_query': "SELECT email FROM users WHERE name = 'Aarav Sharma';",
 'has_problem': False,
 'problem_description': ''}